Extracts frames at 10fps (320×180), runs YOLOv26-seg for vehicle masks and DepthAnything V2 Small for depth maps, and saves outputs as `.npy` files to Google Drive.

In [ ]:
!pip install -q ultralytics transformers accelerate

In [ ]:
# Mount Google Drive and set up paths
from google.colab import drive

drive.mount("/content/drive")

import os

ROOT = "/content/drive/MyDrive/IDS705/nexar"

PATHS = {
    "train_videos": os.path.join(ROOT, "videos", "train"),
    "test_videos": os.path.join(ROOT, "videos", "test"),
    "frames_train": os.path.join(ROOT, "frames", "train"),
    "frames_test": os.path.join(ROOT, "frames", "test"),
    "seg_train": os.path.join(ROOT, "segmentation", "train"),
    "seg_test": os.path.join(ROOT, "segmentation", "test"),
    "depth_train": os.path.join(ROOT, "depth", "train"),
    "depth_test": os.path.join(ROOT, "depth", "test"),
}

# Set HF_HOME early — before transformers is imported anywhere
os.environ["HF_HOME"] = os.path.join(ROOT, "models", "hf_cache")

# Create all output folders if they don't exist
for path in PATHS.values():
    os.makedirs(path, exist_ok=True)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print("Drive mounted. Folder structure:")
for key, path in PATHS.items():
    print(f"  {key:20s} -> {path}")
print(f"  {'HF_HOME':20s} -> {os.environ['HF_HOME']}")

In [ ]:
# Load models (cached to Drive)
import shutil
import torch

MODELS_DIR = os.path.join(ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ── YOLO ──────────────────────────────────────────────────────────────────────
from ultralytics import YOLO

yolo_drive_path = os.path.join(MODELS_DIR, "yolo26n-seg.pt")

if not os.path.exists(yolo_drive_path):
    print("Downloading YOLO — will cache to Drive...")
    _tmp = YOLO("yolo26n-seg.pt")
    shutil.copy("yolo26n-seg.pt", yolo_drive_path)
    yolo = _tmp
    print(f"YOLO cached to: {yolo_drive_path}")
else:
    print("Loading YOLO from Drive cache...")
    yolo = YOLO(yolo_drive_path)

yolo.to(DEVICE)
print("YOLO loaded")

# ── DepthAnything V2 Small (fp16) ─────────────────────────────────────────────
from transformers import pipeline

depth_pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=0 if DEVICE == "cuda" else -1,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
print("DepthAnything V2 Small loaded (fp16)")

In [ ]:
# Helper functions
import cv2
import numpy as np
from PIL import Image

TARGET_FPS = 10
TARGET_W, TARGET_H = 320, 180

# YOLO class IDs for vehicles (COCO): car, motorcycle, bus, truck
VEHICLE_CLASS_IDS = {2, 3, 5, 7}


def extract_frames(video_path):
    """Extract frames at TARGET_FPS, resize to TARGET_W x TARGET_H.
    Returns np.ndarray of shape [N, H, W, 3], dtype uint8."""
    cap = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    interval = max(1, round(src_fps / TARGET_FPS))

    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % interval == 0:
            frame = cv2.resize(frame, (TARGET_W, TARGET_H))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        idx += 1
    cap.release()
    return np.stack(frames).astype(np.uint8)  # [N, H, W, 3]


def extract_seg_masks(frames, batch_size=32):
    """Run YOLOv26-seg on frames in batches, return binary vehicle masks.
    Returns np.ndarray of shape [N, H, W], dtype uint8."""
    masks = []
    for i in range(0, len(frames), batch_size):
        batch = list(frames[i : i + batch_size])
        results = yolo(batch, verbose=False)
        for result in results:
            mask = np.zeros((TARGET_H, TARGET_W), dtype=np.uint8)
            if result.masks is not None:
                for seg_mask, cls_id in zip(result.masks.data, result.boxes.cls):
                    if int(cls_id) in VEHICLE_CLASS_IDS:
                        m = seg_mask.cpu().numpy()
                        m = cv2.resize(
                            m, (TARGET_W, TARGET_H), interpolation=cv2.INTER_NEAREST
                        )
                        mask = np.maximum(mask, (m > 0.5).astype(np.uint8))
            masks.append(mask)
    return np.stack(masks)  # [N, H, W]


def extract_depth_maps(frames, batch_size=32):
    """Run DepthAnything V2 on frames in batches, return normalised depth maps.
    Returns np.ndarray of shape [N, H, W], dtype float16."""
    pil_frames = [Image.fromarray(f) for f in frames]
    depth_maps = []

    for i in range(0, len(pil_frames), batch_size):
        batch = pil_frames[i : i + batch_size]
        results = depth_pipe(batch)
        for result in results:
            depth = np.array(result["depth"], dtype=np.float32)
            depth = cv2.resize(depth, (TARGET_W, TARGET_H))
            d_min, d_max = depth.min(), depth.max()
            if d_max > d_min:
                depth = (depth - d_min) / (d_max - d_min)
            depth_maps.append(depth.astype(np.float16))

    return np.stack(depth_maps)  # [N, H, W]


print("Helper functions defined.")

In [ ]:
# Sanity check on a single video
import matplotlib.pyplot as plt

# Pick the first train video
test_video_id = sorted(os.listdir(PATHS["train_videos"]))[0].replace(".mp4", "")
test_video_path = os.path.join(PATHS["train_videos"], f"{test_video_id}.mp4")
print(f"Testing on: {test_video_path}")

# Extract
frames = extract_frames(test_video_path)
print(f"Frames:       {frames.shape}  dtype={frames.dtype}")

masks = extract_seg_masks(frames)
print(f"Seg masks:    {masks.shape}  dtype={masks.dtype}")

depths = extract_depth_maps(frames)
print(f"Depth maps:   {depths.shape}  dtype={depths.dtype}")

# Visualise frame 0
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].imshow(frames[0])
axes[0].set_title("RGB frame")
axes[1].imshow(masks[0], cmap="gray")
axes[1].set_title("Seg mask")
axes[2].imshow(depths[0], cmap="plasma")
axes[2].set_title("Depth map")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Main preprocessing loop
from tqdm import tqdm


def process_split(split):
    video_dir = PATHS[f"{split}_videos"]
    frames_dir = PATHS[f"frames_{split}"]
    seg_dir = PATHS[f"seg_{split}"]
    depth_dir = PATHS[f"depth_{split}"]

    video_ids = sorted(
        f.replace(".mp4", "") for f in os.listdir(video_dir) if f.endswith(".mp4")
    )
    print(f"\n[{split}] {len(video_ids)} videos found")

    skipped = 0
    for vid_id in tqdm(video_ids, desc=split):
        frames_path = os.path.join(frames_dir, f"{vid_id}.npy")
        seg_path = os.path.join(seg_dir, f"{vid_id}.npy")
        depth_path = os.path.join(depth_dir, f"{vid_id}.npy")

        # Resume: skip if all three outputs already exist
        if (
            os.path.exists(frames_path)
            and os.path.exists(seg_path)
            and os.path.exists(depth_path)
        ):
            skipped += 1
            continue

        video_path = os.path.join(video_dir, f"{vid_id}.mp4")

        # Step 1 — frames
        if not os.path.exists(frames_path):
            frames = extract_frames(video_path)
            np.save(frames_path, frames)
        else:
            frames = np.load(frames_path)

        # Step 2 — segmentation
        if not os.path.exists(seg_path):
            masks = extract_seg_masks(frames)
            np.save(seg_path, masks)

        # Step 3 — depth
        if not os.path.exists(depth_path):
            depths = extract_depth_maps(frames)
            np.save(depth_path, depths)

    print(f"[{split}] Done. Skipped (already processed): {skipped}")


process_split("train")
process_split("test")